# PCA (Principal Component Analysis) - Hands-On Exercises

## Scenario: Visualizing SPG Outlet Data in 2D

You have 5-dimensional outlet data (sales, SKU count, visit frequency, OOS rate, basket size). You want to visualize it in 2D to understand patterns and show it to your team.

**Goal:** Use PCA to reduce 5D to 2D while preserving as much information as possible.

## Exercise 1: Load and Explore Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load the dataset
def load_data(filepath='../01_kmeans/datasets/spg_outlets.csv'):
    data = []
    with open(filepath, 'r') as f:
        headers = f.readline().strip().split(',')
        for line in f:
            data.append(line.strip().split(','))
    
    features = []
    for row in data:
        features.append([
            float(row[4]),  # monthly_sales_idr
            float(row[5]),  # sku_count
            float(row[6]),  # visit_freq_per_month
            float(row[7]),  # oos_rate
            float(row[8])   # avg_basket_idr
        ])
    return np.array(features)

X = load_data()
feature_names = ['Monthly Sales', 'SKU Count', 'Visit Freq', 'OOS Rate', 'Avg Basket']

print(f"Loaded {len(X)} outlets with {X.shape[1]} features")
print(f"\nFeature ranges (critical for understanding why scaling matters!):")
for i, name in enumerate(feature_names):
    print(f"  {name:20s}: min={X[:, i].min():.2f}, max={X[:, i].max():.2f}, range={X[:, i].max()-X[:, i].min():.2f}")

**Question:** Without scaling, which feature would dominate PCA? Why?

## Exercise 2: Scale Features and Fit PCA

In [ ]:
# TODO: Scale features to zero mean and unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✓ Features scaled")
print(f"  Mean: {X_scaled.mean(axis=0)}")
print(f"  Std: {X_scaled.std(axis=0)}")

# Fit PCA with ALL components first
pca_all = PCA()
pca_all.fit(X_scaled)

print(f"\nPCA fitted with {pca_all.n_components_} components")

## Exercise 3: Analyze Explained Variance

In [ ]:
print("Variance explained by each component:\n")

for i in range(pca_all.n_components_):
    var = pca_all.explained_variance_[i]
    ratio = pca_all.explained_variance_ratio_[i]
    cumsum = pca_all.explained_variance_ratio_[:i+1].sum()
    
    print(f"PC{i+1}: variance={var:6.2f}, ratio={ratio:.3f} ({ratio*100:5.1f}%), cumulative={cumsum:.3f} ({cumsum*100:5.1f}%)")

# Plot
plt.figure(figsize=(12, 5))

# Scree plot (individual variance)
plt.subplot(1, 2, 1)
plt.bar(range(1, pca_all.n_components_+1), pca_all.explained_variance_ratio_, alpha=0.7)
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained Ratio')
plt.title('Scree Plot')
plt.grid(True, alpha=0.3, axis='y')

# Cumulative variance
plt.subplot(1, 2, 2)
cumsum = pca_all.explained_variance_ratio_.cumsum()
plt.plot(range(1, pca_all.n_components_+1), cumsum, 'bo-', linewidth=2, markersize=10)
plt.axhline(y=0.90, color='red', linestyle='--', label='90% threshold')
plt.axhline(y=0.95, color='orange', linestyle='--', label='95% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Variance Explained')
plt.title('Cumulative Variance')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find how many components for 90%
n_90 = np.argmax(cumsum >= 0.90) + 1
n_95 = np.argmax(cumsum >= 0.95) + 1
print(f"\nTo capture 90% of variance, need {n_90} components")
print(f"To capture 95% of variance, need {n_95} components")

**Question:** How much variance do the first 2 components capture? Is that enough for visualization?

## Exercise 4: Reduce to 2D and Visualize

In [ ]:
# TODO: Fit PCA with 2 components and transform data
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Reduced from {X_scaled.shape[1]}D to {X_pca.shape[1]}D")
print(f"Variance captured: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Visualize in 2D
plt.figure(figsize=(10, 8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=50)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.title('Outlet Data in 2D PCA Space', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nYou can now see the structure of 5D data in 2D!")

## Exercise 5: Understand Feature Contributions (Loadings)

In [ ]:
# Principal component loadings show which original features contribute to each PC
print("Feature contributions to each principal component:\n")

for i in range(2):
    print(f"PC{i+1} (explains {pca.explained_variance_ratio_[i]*100:.1f}% variance):")
    loadings = pca.components_[i]
    
    # Sort by absolute magnitude
    sorted_indices = np.argsort(np.abs(loadings))[::-1]
    
    for idx in sorted_indices:
        contribution = loadings[idx]
        print(f"  {feature_names[idx]:20s}: {contribution:+.3f}")
    print()

# Visualize loadings as biplot
plt.figure(figsize=(10, 8))

# Plot data points
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.3, s=30, c='lightblue', edgecolors='gray')

# Plot feature vectors
scale = 3  # Scale arrows for visibility
for i, feature in enumerate(feature_names):
    plt.arrow(0, 0, 
             pca.components_[0, i] * scale,
             pca.components_[1, i] * scale,
             head_width=0.2, head_length=0.2, fc='red', ec='red', linewidth=2)
    plt.text(pca.components_[0, i] * scale * 1.15,
            pca.components_[1, i] * scale * 1.15,
            feature, fontsize=11, fontweight='bold', ha='center', va='center')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
plt.title('PCA Biplot: Data + Feature Contributions', fontsize=14, fontweight='bold')
plt.axhline(y=0, color='black', linewidth=0.5)
plt.axvline(x=0, color='black', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Question:** Which original features contribute most to PC1? What does PC1 represent?

## Exercise 6: Visualize K-Means Clusters in PCA Space

In [ ]:
# Cluster in the ORIGINAL scaled space (5D)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

print(f"✓ Clustered into 4 segments")
unique, counts = np.unique(labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"  Cluster {cluster_id}: {count} outlets")

# Visualize clusters in 2D PCA space
plt.figure(figsize=(12, 8))

colors = ['red', 'blue', 'green', 'orange']
for i in range(4):
    cluster_mask = labels == i
    plt.scatter(X_pca[cluster_mask, 0], X_pca[cluster_mask, 1],
               c=colors[i], label=f'Cluster {i}', alpha=0.6, s=50)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
plt.title('K-Means Clusters Visualized in 2D PCA Space', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nThis is the power of PCA: visualize 5D clusters in 2D!")

## Exercise 7: Reconstruction and Information Loss

In [ ]:
# Reconstruct data from 2D back to 5D
X_reconstructed = pca.inverse_transform(X_pca)

# Compute reconstruction error
mse = np.mean((X_scaled - X_reconstructed) ** 2)
print(f"Reconstruction error (MSE) with 2 components: {mse:.4f}")

# Compare with different numbers of components
print("\nReconstruction error vs. number of components:\n")
for n_comp in range(1, 6):
    pca_temp = PCA(n_components=n_comp)
    X_temp = pca_temp.fit_transform(X_scaled)
    X_recon = pca_temp.inverse_transform(X_temp)
    mse = np.mean((X_scaled - X_recon) ** 2)
    var = pca_temp.explained_variance_ratio_.sum()
    print(f"  {n_comp} components: MSE={mse:.6f}, variance={var:.3f} ({var*100:.1f}%)")

print("\nNote: With all 5 components, reconstruction is perfect (MSE ≈ 0)")

**Question:** Is the reconstruction error acceptable with 2 components? What's the trade-off?

## Exercise 8: Compare Different Scalings (Bonus)

In [ ]:
# Compare PCA with and without scaling
print("Explained variance ratio comparison:\n")

# Without scaling (BAD!)
pca_no_scale = PCA(n_components=2)
pca_no_scale.fit(X)
print("Without scaling:")
print(f"  PC1: {pca_no_scale.explained_variance_ratio_[0]:.3f}")
print(f"  PC2: {pca_no_scale.explained_variance_ratio_[1]:.3f}")
print(f"  Total: {pca_no_scale.explained_variance_ratio_.sum():.3f}")

# With scaling (GOOD!)
print("\nWith scaling:")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.3f}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.3f}")
print(f"  Total: {pca.explained_variance_ratio_.sum():.3f}")

print("\n⚠️  Always scale features before PCA!")
print("Without scaling, features with large ranges dominate.")

## Summary

**What you learned:**
1. ✅ PCA reduces dimensionality while preserving variance
2. ✅ Feature scaling is CRITICAL for PCA
3. ✅ Use explained variance ratio to choose n_components
4. ✅ Loadings show which original features contribute to each PC
5. ✅ PCA is excellent for visualizing high-dimensional data
6. ✅ Reconstruction error tells you how much information was lost

**Next steps:**
- Try different numbers of components and compare
- Use PCA before other algorithms (clustering, classification) to speed them up
- Explore t-SNE for non-linear dimensionality reduction
- Interpret what each principal component means in business terms